In [1]:
# =========================================================
# IMPORTS
# =========================================================

import numpy as np
import pandas as pd
import faiss

from tqdm import tqdm

# =========================================================
# LOAD DATA
# =========================================================

EMBEDDINGS_PATH = "embeddings.npy"
METADATA_PATH = "metadata_with_embeddings.csv"

embeddings = np.load(EMBEDDINGS_PATH)

metadata = pd.read_csv(METADATA_PATH)

print("=" * 60)

print("DATA LOADED")

print("=" * 60)

print()

print("Embeddings Shape:", embeddings.shape)

# =========================================================
# NORMALIZE EMBEDDINGS
# =========================================================

embeddings = embeddings.astype("float32")

norms = np.linalg.norm(
    embeddings,
    axis=1,
    keepdims=True
)

embeddings = embeddings / norms

print()

print("Embeddings Normalized")

# =========================================================
# SPLITS
# =========================================================

query_df = metadata[
    metadata["split"] == "query"
].reset_index(drop=True)

gallery_df = metadata[
    metadata["split"] == "gallery"
].reset_index(drop=True)

print()

print("Queries:", len(query_df))

print("Gallery:", len(gallery_df))

# =========================================================
# GALLERY EMBEDDINGS
# =========================================================

gallery_indices = gallery_df[
    "embedding_index"
].values

gallery_embeddings = embeddings[
    gallery_indices
]

gallery_embeddings = gallery_embeddings.astype(
    "float32"
)

print()

print(
    "Gallery Embeddings Shape:",
    gallery_embeddings.shape
)

# =========================================================
# BUILD HNSW FAISS INDEX
# =========================================================

dim = gallery_embeddings.shape[1]

gallery_index = faiss.IndexHNSWFlat(
    dim,
    32
)

gallery_index.hnsw.efConstruction = 40
gallery_index.hnsw.efSearch = 16

gallery_index.add(gallery_embeddings)

print()

print("HNSW FAISS Index Built")

print(
    "Total Gallery Vectors:",
    gallery_index.ntotal
)

# =========================================================
# METRICS
# =========================================================

def recall_at_k(relevant, retrieved, k):

    retrieved_k = retrieved[:k]

    return int(
        any(r in relevant for r in retrieved_k)
    )


def average_precision_at_k(
    relevant,
    retrieved,
    k
):

    retrieved_k = retrieved[:k]

    score = 0

    hits = 0

    for i, item in enumerate(retrieved_k):

        if item in relevant:

            hits += 1

            score += hits / (i + 1)

    if len(relevant) == 0:
        return 0

    return score / len(relevant)


def dcg_at_k(
    relevant,
    retrieved,
    k
):

    retrieved_k = retrieved[:k]

    dcg = 0

    for i, item in enumerate(retrieved_k):

        if item in relevant:

            dcg += 1 / np.log2(i + 2)

    return dcg


def ndcg_at_k(
    relevant,
    retrieved,
    k
):

    ideal = dcg_at_k(
        relevant,
        relevant,
        min(k, len(relevant))
    )

    if ideal == 0:
        return 0

    return dcg_at_k(
        relevant,
        retrieved,
        k
    ) / ideal

# =========================================================
# EVALUATION
# =========================================================

recalls = {
    5: [],
    10: [],
    15: []
}

aps = {
    5: [],
    10: [],
    15: []
}

ndcgs = {
    5: [],
    10: [],
    15: []
}

print()

print("=" * 60)

print("RUNNING EVALUATION")

print("=" * 60)

print()

# =========================================================
# QUERY LOOP
# =========================================================

for i in tqdm(range(len(query_df))):

    query_item_id = query_df.iloc[i][
        "item_id"
    ]

    relevant = list(

        gallery_df[
            gallery_df["item_id"]
            == query_item_id
        ]["item_id"]

    )

    query_embedding_idx = query_df.iloc[i][
        "embedding_index"
    ]

    query_vector = embeddings[
        query_embedding_idx
    ].reshape(1, -1)

    scores, indices = gallery_index.search(
        query_vector,
        15
    )

    retrieved = [

        gallery_df.iloc[idx]["item_id"]

        for idx in indices[0]

    ]

    for k in [5, 10, 15]:

        recalls[k].append(

            recall_at_k(
                relevant,
                retrieved,
                k
            )

        )

        aps[k].append(

            average_precision_at_k(
                relevant,
                retrieved,
                k
            )

        )

        ndcgs[k].append(

            ndcg_at_k(
                relevant,
                retrieved,
                k
            )

        )

# =========================================================
# FINAL RESULTS
# =========================================================

print()

print("=" * 60)

print("FINAL METRICS")

print("=" * 60)

print()

for k in [5, 10, 15]:

    print(
        f"Recall@{k}:",
        round(np.mean(recalls[k]), 4)
    )

    print(
        f"mAP@{k}:",
        round(np.mean(aps[k]), 4)
    )

    print(
        f"NDCG@{k}:",
        round(np.mean(ndcgs[k]), 4)
    )

    print()

print("=" * 60)

print("EVALUATION COMPLETE")

print("=" * 60)

DATA LOADED

Embeddings Shape: (55313, 512)

Embeddings Normalized

Queries: 1330
Gallery: 15144

Gallery Embeddings Shape: (15144, 512)

HNSW FAISS Index Built
Total Gallery Vectors: 15144

RUNNING EVALUATION



100%|██████████| 1330/1330 [00:02<00:00, 634.13it/s]


FINAL METRICS

Recall@5: 0.4105
mAP@5: 0.1151
NDCG@5: 0.1875

Recall@10: 0.4639
mAP@10: 0.1221
NDCG@10: 0.1865

Recall@15: 0.491
mAP@15: 0.1252
NDCG@15: 0.1912

EVALUATION COMPLETE
